In [1]:
import sys
import os
# Add parent directory to path to import tradingutils modules
sys.path.insert(0, os.path.abspath('..'))

from kalshi_utils.client_wrapper import *

kalshi_wrapped = KalshiWrapped()

nba_live_markets = kalshi_wrapped.GetALLNBAMarkets(status="open")
nhl_live_markets = kalshi_wrapped.GetAllNHLMarkets(status="open")
college_markets = kalshi_wrapped.GetALLNCAAMBMarkets(status="open")
tennis_markets = kalshi_wrapped.GetALLTennisMarkets(status="open")

pairs = kalshi_wrapped.GetMarketPairs(nba_live_markets)

Client Created


In [2]:
pairs

[(Market(ticker='KXNBAGAME-26JAN22MIAPOR-POR', event_ticker='KXNBAGAME-26JAN22MIAPOR', market_type='binary', title='Miami at Portland Winner?', subtitle='', yes_sub_title='Portland', no_sub_title='Portland', created_time=datetime.datetime(2026, 1, 20, 14, 3, 4, 696174, tzinfo=TzInfo(0)), open_time=datetime.datetime(2026, 1, 20, 19, 7, tzinfo=TzInfo(0)), close_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), expected_expiration_time=datetime.datetime(2026, 1, 23, 6, 0, tzinfo=TzInfo(0)), expiration_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), latest_expiration_time=datetime.datetime(2026, 2, 6, 3, 0, tzinfo=TzInfo(0)), settlement_timer_seconds=30, status='active', response_price_units='usd_cent', yes_bid=54, yes_bid_dollars='0.5400', yes_ask=55, yes_ask_dollars='0.5500', no_bid=45, no_bid_dollars='0.4500', no_ask=46, no_ask_dollars='0.4600', last_price=55, last_price_dollars='0.5500', volume=287, volume_24h=270, result='', can_close_early=True, open_interest=28

In [3]:
import time
import threading
from datetime import datetime
import uuid
import os
import logging
import math


class MarketMakerBot:
    """
    Single-level market maker for a Kalshi-style binary market.

    Assumptions:
      - Prices are in [0, 1] dollars (1–99 cents ticks).
      - create_order uses: ticker, action in {"buy","sell"}, side in {"yes","no"},
        count, type="limit", and yes_price/no_price in cents.
      - get_order returns an object with .order.status. If it also has
        .order.filled_count (or similar), partial fills are reconciled.
    """

    TICK = 0.01

    def __init__(
        self,
        client,
        market,
        side: str = "yes",
        spread_bps: int = 50,
        size_per_level: int = 1,
        max_position: int = 10,
        stop_loss_pct: float = -2.0,
        take_profit_pct: float = 3.0,
        skew_on_inventory: bool = True,
        skew_bps_per_contract: int = 10,
        check_interval_s: float = 2.0,
        maker_fee_pct: float = 0.0,
        taker_fee_pct: float = 7.0,
        max_order_age_s: float = 10.0,
        logger: logging.Logger = None,
    ):
        if side not in ("yes", "no"):
            raise ValueError("side must be 'yes' or 'no'")

        self.client = client
        self.market = market
        self.ticker = market.model_dump().get("ticker") if hasattr(market, "model_dump") else getattr(market, "ticker", str(market))
        self.side = side

        # config
        self.spread = spread_bps / 10000.0
        self.size_per_level = int(size_per_level)
        self.max_position = int(max_position)
        self.stop_loss = stop_loss_pct / 100.0
        self.take_profit = take_profit_pct / 100.0
        self.skew_on_inventory = bool(skew_on_inventory)
        self.skew_per_contract = (skew_bps_per_contract / 10000.0)
        self.check_interval_s = float(check_interval_s)
        self.maker_fee = maker_fee_pct / 100.0
        self.taker_fee = taker_fee_pct / 100.0
        self.max_order_age_s = float(max_order_age_s)

        # logging
        if logger is None:
            log_dir = "logs"
            os.makedirs(log_dir, exist_ok=True)
            session_id = str(uuid.uuid4())[:8]
            date_str = datetime.now().strftime("%Y-%m-%d")
            log_path = os.path.join(log_dir, f"mm_{date_str}_{session_id}.log")

            self.logger = logging.getLogger(f"mm_{session_id}")
            self.logger.setLevel(logging.DEBUG)
            self.logger.propagate = False
            if self.logger.handlers:
                self.logger.handlers.clear()

            fh = logging.FileHandler(log_path)
            fh.setLevel(logging.DEBUG)
            fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
            self.logger.addHandler(fh)

            ch = logging.StreamHandler()
            ch.setLevel(logging.INFO)
            ch.setFormatter(logging.Formatter("%(message)s"))
            self.logger.addHandler(ch)

            self.logger.info(f"=== MarketMakerBot started: {log_path} ===")
        else:
            self.logger = logger

        # state (guard with lock)
        self._lock = threading.Lock()

        self.position = 0                    # signed contracts; + long, - short
        self.avg_entry_price = 0.0           # avg price of open position (net of entry fees)
        self.realized_pnl = 0.0              # after fees
        self.total_fees_paid = 0.0
        self.total_filled = 0

        self.margin_used = 0.0               # current worst-case loss estimate
        self.peak_margin_used = 0.0

        # active orders keyed by action ("buy"/"sell")
        # each: {id, price_cents, count, t, filled_so_far}
        self.active_orders = {}

        self.stop_evt = threading.Event()
        self.running = False
        self.thread = None

        self.logger.info("MarketMakerBot initialized:")
        self.logger.info(f"  Ticker: {self.ticker}, Side: {self.side}")
        self.logger.info(f"  Spread: {spread_bps} bps | Size: {self.size_per_level} | MaxPos: ±{self.max_position}")
        self.logger.info(f"  Stop: {stop_loss_pct:.2f}% | Take: {take_profit_pct:.2f}%")
        self.logger.info(f"  Fees: maker={maker_fee_pct:.4f}% taker={taker_fee_pct:.4f}%")
        self.logger.info(f"  Skew: {self.skew_on_inventory} ({skew_bps_per_contract} bps/contract)")
        self.logger.info(f"  max_order_age_s={self.max_order_age_s}")

    # ---------- lifecycle ----------

    def start(self):
        if self.running:
            self.logger.warning("Bot already running")
            return self
        self.running = True
        self.stop_evt.clear()
        self.thread = threading.Thread(target=self._run, daemon=True, name=f"MM-{self.ticker}")
        self.thread.start()
        self.logger.info("Market maker bot started")
        return self

    def stop(self, join: bool = True):
        """
        Signal stop; optionally join thread if called externally.
        Safe to call from inside the worker thread (will not self-join).
        """
        self.stop_evt.set()
        if join and self.thread and threading.current_thread() is not self.thread:
            self.thread.join(timeout=5.0)

    # ---------- market data / quoting ----------

    def _get_mid_price(self):
        try:
            market_response = self.client.get_market(self.ticker)
            data = market_response.market.model_dump() if hasattr(market_response, "market") else {}

            def sf(x, default=None):
                if x is None:
                    return default
                try:
                    return float(x)
                except (TypeError, ValueError):
                    return default

            if self.side == "yes":
                bid = sf(data.get("yes_bid_dollars"), None)
                ask = sf(data.get("yes_ask_dollars"), None)
            else:
                bid = sf(data.get("no_bid_dollars"), None)
                ask = sf(data.get("no_ask_dollars"), None)

            if bid is not None and ask is not None and 0 < bid < ask < 1:
                mid = (bid + ask) / 2.0
            else:
                # fallback; prefer a sane mid rather than crashing
                mid = 0.5
                bid = bid if bid is not None else 0.0
                ask = ask if ask is not None else 1.0

            return mid, float(bid), float(ask)
        except Exception as e:
            self.logger.error(f"Error getting mid price: {e}")
            return 0.5, 0.0, 1.0

    def _quantize_quotes(self, bid: float, ask: float):
        """
        Floor bid to cents, ceil ask to cents. Enforce 1-cent tick and ask>bid.
        Clamp to [1,99] cents.
        """
        bid_c = int(math.floor(bid * 100 + 1e-9))
        ask_c = int(math.ceil(ask * 100 - 1e-9))

        bid_c = max(1, min(98, bid_c))
        ask_c = max(2, min(99, ask_c))

        if ask_c <= bid_c:
            # enforce at least 2 cents total spread if rounding collapses
            mid_c = int(round(((bid + ask) / 2.0) * 100))
            bid_c = max(1, min(98, mid_c - 1))
            ask_c = max(2, min(99, mid_c + 1))
            if ask_c <= bid_c:
                ask_c = min(99, bid_c + 1)

        return bid_c, ask_c

    def _calculate_quotes(self):
        mid, mkt_bid, mkt_ask = self._get_mid_price()

        base_bid = mid - self.spread / 2.0
        base_ask = mid + self.spread / 2.0

        # inventory skew shifts the mid
        with self._lock:
            pos = self.position
        if self.skew_on_inventory and pos != 0:
            skew = pos * self.skew_per_contract
            base_bid -= skew
            base_ask -= skew

        # stay in bounds
        base_bid = max(self.TICK, min(1.0 - self.TICK, base_bid))
        base_ask = max(self.TICK, min(1.0 - self.TICK, base_ask))

        bid_c, ask_c = self._quantize_quotes(base_bid, base_ask)

        # do-not-cross clamp to avoid accidental taker fills
        # buy must be <= (market ask - tick), sell must be >= (market bid + tick)
        if mkt_ask is not None and 0 < mkt_ask < 1:
            max_bid_c = int(round((mkt_ask - self.TICK) * 100))
            bid_c = min(bid_c, max(1, min(98, max_bid_c)))
        if mkt_bid is not None and 0 < mkt_bid < 1:
            min_ask_c = int(round((mkt_bid + self.TICK) * 100))
            ask_c = max(ask_c, max(2, min(99, min_ask_c)))

        if ask_c <= bid_c:
            ask_c = min(99, bid_c + 1)

        return bid_c, ask_c, mid

    def _should_quote(self):
        with self._lock:
            pos = self.position
        can_buy = pos < self.max_position
        can_sell = pos > -self.max_position
        return can_buy, can_sell

    # ---------- fees / risk ----------

    def _fee_rate(self, is_maker: bool):
        return self.maker_fee if is_maker else self.taker_fee

    def _calculate_fee(self, action: str, price: float, quantity: int, is_maker: bool = True):
        """
        Fee modeled as: fee_rate * potential_profit * quantity

        potential_profit per contract:
          - buy at p -> (1 - p)
          - sell at p -> p
        """
        r = self._fee_rate(is_maker)
        p = float(price)
        q = int(quantity)
        p = max(0.0, min(1.0, p))
        pot = (1.0 - p) if action == "buy" else p
        return q * pot * r

    def _recompute_margin(self):
        """
        Approximate worst-case loss (collateral) based on current open position.
          long q @ p: q*p
          short q @ p: q*(1-p)
        """
        with self._lock:
            pos = self.position
            p = self.avg_entry_price

        if pos == 0:
            m = 0.0
        elif pos > 0:
            m = pos * p
        else:
            m = (-pos) * (1.0 - p)

        with self._lock:
            self.margin_used = m
            self.peak_margin_used = max(self.peak_margin_used, m)

    def _calculate_unrealized_pnl(self):
        with self._lock:
            pos = self.position
            entry = self.avg_entry_price

        if pos == 0:
            return 0.0

        mid, _, _ = self._get_mid_price()

        # estimate exit fee as maker (best-effort)
        exit_fee = self._calculate_fee("sell" if pos > 0 else "buy", mid, abs(pos), is_maker=True)

        if pos > 0:
            return pos * (mid - entry) - exit_fee
        else:
            return (-pos) * (entry - mid) - exit_fee

    def get_total_pnl(self):
        with self._lock:
            realized = self.realized_pnl
        return realized + self._calculate_unrealized_pnl()

    def _check_risk_limits(self):
        self._recompute_margin()
        with self._lock:
            denom = self.peak_margin_used

        if denom <= 0:
            return False

        total_pnl = self.get_total_pnl()
        pnl_pct = total_pnl / denom

        if pnl_pct <= self.stop_loss:
            self.logger.warning(f"STOP LOSS hit: {pnl_pct*100:.2f}% (PnL ${total_pnl:.2f}, denom ${denom:.2f})")
            return True
        if pnl_pct >= self.take_profit:
            self.logger.info(f"TAKE PROFIT hit: {pnl_pct*100:.2f}% (PnL ${total_pnl:.2f}, denom ${denom:.2f})")
            return True
        return False

    # ---------- orders ----------

    def _place_order(self, action: str, price_cents: int, quantity: int):
        try:
            price_cents = int(price_cents)
            quantity = int(quantity)
            price_cents = max(1, min(99, price_cents))
            if quantity <= 0:
                return None

            params = {
                "ticker": self.ticker,
                "action": action,
                "side": self.side,
                "count": quantity,
                "type": "limit",
            }
            if self.side == "yes":
                params["yes_price"] = price_cents
            else:
                params["no_price"] = price_cents

            resp = self.client.create_order(**params)
            order_id = getattr(getattr(resp, "order", None), "order_id", None)

            if order_id:
                with self._lock:
                    self.active_orders[action] = {
                        "id": order_id,
                        "price_cents": price_cents,
                        "count": quantity,
                        "t": time.time(),
                        "filled_so_far": 0,
                    }
                self.logger.debug(f"Placed {action} {quantity} @ {price_cents}c (id={order_id})")
            return order_id
        except Exception as e:
            self.logger.error(f"Error placing {action} order: {e}")
            return None

    def _cancel_order(self, action: str):
        with self._lock:
            info = self.active_orders.get(action)
        if not info:
            return
        oid = info["id"]
        try:
            self.client.cancel_order(order_id=oid)
        except Exception as e:
            # treat as non-fatal; order may have filled/expired
            self.logger.debug(f"Cancel {action} order error (id={oid}): {e}")
        finally:
            with self._lock:
                self.active_orders.pop(action, None)

    def _cancel_all_orders(self):
        for action in ("buy", "sell"):
            self._cancel_order(action)

    def _maybe_refresh_order(self, action: str, desired_price_cents: int, desired_count: int):
        """
        Replace only if:
          - no existing order
          - price/size changed
          - order age exceeded
        """
        now = time.time()
        with self._lock:
            info = self.active_orders.get(action)

        if info:
            same_price = (info["price_cents"] == desired_price_cents)
            same_count = (info["count"] == desired_count)
            age = now - info["t"]
            if same_price and same_count and age <= self.max_order_age_s:
                return  # keep resting order

            # otherwise replace
            self._cancel_order(action)

        self._place_order(action, desired_price_cents, desired_count)

    def _check_fills(self):
        """
        Best-effort:
          - If order.status == "filled": apply remaining qty.
          - If order exposes filled_count: apply delta fills.
        """
        with self._lock:
            snapshot = {a: dict(v) for a, v in self.active_orders.items()}

        for action, info in snapshot.items():
            oid = info["id"]
            try:
                resp = self.client.get_order(order_id=oid)
                order = getattr(resp, "order", None)
                if not order:
                    continue

                status = getattr(order, "status", None)

                # attempt partial fill reconciliation
                filled_count = getattr(order, "filled_count", None)
                if filled_count is None:
                    # some APIs use "filled" or "fill_count"
                    filled_count = getattr(order, "fill_count", None)

                if filled_count is not None:
                    filled_count = int(filled_count)
                    prev = int(info.get("filled_so_far", 0))
                    delta = max(0, filled_count - prev)
                    if delta > 0:
                        self._handle_fill(action, info["price_cents"] / 100.0, delta, is_maker=True)
                        with self._lock:
                            if action in self.active_orders:
                                self.active_orders[action]["filled_so_far"] = filled_count

                    if filled_count >= info["count"] or status in ("filled", "canceled", "expired"):
                        with self._lock:
                            self.active_orders.pop(action, None)
                    continue

                # fallback: only handle full fills
                if status == "filled":
                    remaining = info["count"] - int(info.get("filled_so_far", 0))
                    if remaining > 0:
                        self._handle_fill(action, info["price_cents"] / 100.0, remaining, is_maker=True)
                    with self._lock:
                        self.active_orders.pop(action, None)
                elif status in ("canceled", "expired"):
                    with self._lock:
                        self.active_orders.pop(action, None)

            except Exception as e:
                self.logger.debug(f"Error checking order {oid}: {e}")

    def _handle_fill(self, action: str, price: float, quantity: int, is_maker: bool = True):
        """
        Update position, avg_entry, realized pnl with consistent fee allocation.
        avg_entry is stored net of entry fees:
          - buy increases cost basis => effective entry = price + fee/q
          - sell increases proceeds => effective entry = price - fee/q (for shorts)
        """
        q = int(quantity)
        if q <= 0:
            return

        fee = self._calculate_fee(action, price, q, is_maker=is_maker)
        fee_per = fee / q if q else 0.0

        with self._lock:
            self.total_fees_paid += fee
            pos = self.position
            entry = self.avg_entry_price

        eff_price = (price + fee_per) if action == "buy" else (price - fee_per)

        realized_delta = 0.0
        new_pos = pos
        new_entry = entry

        if action == "buy":
            if pos >= 0:
                # add/increase long
                new_pos = pos + q
                new_entry = ((pos * entry) + (q * eff_price)) / new_pos if new_pos != 0 else eff_price
            else:
                # cover short
                cover = min(q, -pos)
                realized_delta += cover * (entry - price) - (cover * fee_per)
                new_pos = pos + cover  # less negative

                rem = q - cover
                if rem > 0:
                    # flip to long on remaining
                    new_pos = rem
                    new_entry = eff_price
                elif new_pos == 0:
                    new_entry = 0.0

        else:  # sell
            if pos <= 0:
                # add/increase short
                new_pos = pos - q
                short_qty_old = -pos
                short_qty_new = -new_pos
                new_entry = ((short_qty_old * entry) + (q * eff_price)) / short_qty_new if short_qty_new != 0 else eff_price
            else:
                # sell/reduce long
                sold = min(q, pos)
                realized_delta += sold * (price - entry) - (sold * fee_per)
                new_pos = pos - sold

                rem = q - sold
                if rem > 0:
                    # flip to short on remaining
                    new_pos = -rem
                    new_entry = eff_price
                elif new_pos == 0:
                    new_entry = 0.0

        with self._lock:
            self.position = new_pos
            self.avg_entry_price = new_entry
            self.realized_pnl += realized_delta
            self.total_filled += q

        self._recompute_margin()

        ts = datetime.now().strftime("%H:%M:%S")
        unreal = self._calculate_unrealized_pnl()
        total = self.get_total_pnl()
        self.logger.info(
            f"[{ts}] FILL {action.upper():4s} {q:>3d} @ {price:.2f} | fee {fee:.4f} | "
            f"pos {new_pos:+d} | pnl {total:.2f} (real {self.realized_pnl:.2f}, unr {unreal:.2f})"
        )

    # ---------- main loop ----------

    def _run(self):
        try:
            while not self.stop_evt.is_set():
                self._check_fills()

                if self._check_risk_limits():
                    # signal stop and exit loop; do NOT self-join
                    self.stop_evt.set()
                    break

                bid_c, ask_c, mid = self._calculate_quotes()
                can_buy, can_sell = self._should_quote()

                if can_buy:
                    self._maybe_refresh_order("buy", bid_c, self.size_per_level)
                else:
                    self._cancel_order("buy")

                if can_sell:
                    self._maybe_refresh_order("sell", ask_c, self.size_per_level)
                else:
                    self._cancel_order("sell")

                with self._lock:
                    pos = self.position
                total_pnl = self.get_total_pnl()
                self.logger.debug(
                    f"Quotes bid={bid_c}c ask={ask_c}c mid={mid:.3f} | pos {pos:+d}/{self.max_position} | pnl {total_pnl:.2f}"
                )

                time.sleep(self.check_interval_s)

        except Exception as e:
            self.logger.error(f"Fatal error in loop: {e}", exc_info=True)
        finally:
            # orderly shutdown
            self._cancel_all_orders()
            self.running = False
            self._print_summary()

    def _print_summary(self):
        total = self.get_total_pnl()
        with self._lock:
            peak = self.peak_margin_used
            fees = self.total_fees_paid
            filled = self.total_filled
            pos = self.position

        roi = (total / peak * 100.0) if peak > 0 else 0.0

        print("\n" + "=" * 60)
        print("MARKET MAKER SUMMARY")
        print("=" * 60)
        print(f"Ticker: {self.ticker} ({self.side})")
        print(f"Total Contracts Filled: {filled}")
        print(f"Final Position: {pos:+d}")
        print(f"Peak Margin Used (worst-case loss): ${peak:.2f}")
        print(f"Total Fees Paid: ${fees:.2f}")
        print(f"Total P&L (after fees): ${total:.2f} ({roi:.2f}%)")
        print(f"Realized P&L: ${self.realized_pnl:.2f}")
        print(f"Unrealized P&L (est): ${self._calculate_unrealized_pnl():.2f}")
        print("=" * 60 + "\n")


## Market Maker Bot Configuration

Select a market and configure the bot parameters below:

**Key Parameters:**
- `spread_bps`: Spread in basis points (50 bps = 0.5% = $0.005)
- `size_per_level`: Number of contracts to quote on each side
- `max_position`: Maximum inventory (±contracts). Bot won't buy if at +max, won't sell if at -max
- `stop_loss_pct`: Stop trading if return drops to this % (e.g., -2.0 = -2%)
- `take_profit_pct`: Stop trading if return reaches this % (e.g., 3.0 = +3%)
- `skew_on_inventory`: If True, skew quotes when inventory builds up
  - Long inventory → lower quotes to encourage selling
  - Short inventory → raise quotes to encourage buying
- `skew_bps_per_contract`: How much to adjust quotes per contract of inventory
- `check_interval_s`: How often to update quotes (seconds)

In [51]:
# Step 1: Select a market from the available pairs
# Example: Use the first college basketball market pair
if college_markets:
    pairs = kalshi_wrapped.GetMarketPairs(college_markets)
    if pairs:
        # Show first 3 pairs
        print("Available market pairs:")
        for i, pair in enumerate(pairs[:3]):
            m1, m2 = pair
            print(f"\n{i+1}. {m1.model_dump()['ticker']}")
            print(f"   {m1.model_dump()['title']}")
            print(f"   {m2.model_dump()['ticker']}")
            print(f"   {m2.model_dump()['title']}")
        
        # Select the first market's first side
        selected_market = pairs[0][0]
        print(f"\n✓ Selected market: {selected_market.model_dump()['ticker']}")
        print(f"   Title: {selected_market.model_dump()['title']}")
else:
    print("No college markets available. Try NBA or NHL markets.")

Available market pairs:

1. KXNCAAMBGAME-26JAN23CSBHAW-HAW
   Cal State Bakersfield at Hawai'i Winner?
   KXNCAAMBGAME-26JAN23CSBHAW-CSB
   Cal State Bakersfield at Hawai'i Winner?

2. KXNCAAMBGAME-26JAN22UCIUCRV-UCRV
   UC Irvine at UC Riverside Winner?
   KXNCAAMBGAME-26JAN22UCIUCRV-UCI
   UC Irvine at UC Riverside Winner?

3. KXNCAAMBGAME-26JAN22LBSUCSF-LBSU
   Long Beach St. at Cal State Fullerton Winner?
   KXNCAAMBGAME-26JAN22LBSUCSF-CSF
   Long Beach St. at Cal State Fullerton Winner?

✓ Selected market: KXNCAAMBGAME-26JAN23CSBHAW-HAW
   Title: Cal State Bakersfield at Hawai'i Winner?


In [4]:
# Manual market selection (if needed)
# Find a specific market from the available markets
event_name = "kxatpmatch-26jan20zhemou".upper()
selected_market = kalshi_wrapped.GetMarketByTicker(tennis_markets, event_name)
print(f"Selected: {selected_market.model_dump()['ticker']}")

Selected: KXATPMATCH-26JAN20ZHEMOU-ZHE


In [54]:
# Verify the selected market is valid and open
try:
    ticker = selected_market.model_dump()['ticker']
    status = selected_market.model_dump().get('status', 'unknown')
    title = selected_market.model_dump().get('title', 'N/A')
    
    print(f"✓ Selected Market:")
    print(f"  Ticker: {ticker}")
    print(f"  Title: {title}")
    print(f"  Status: {status}")
    
    # Try to get fresh market data
    market_check = kalshi_wrapped.GetClient().get_market(ticker)
    data = market_check.market.model_dump()
    
    # Handle string or numeric values
    def safe_float(val):
        if val is None:
            return 0.0
        try:
            return float(val)
        except (ValueError, TypeError):
            return 0.0
    
    yes_bid = safe_float(data.get('yes_bid_dollars', 0))
    yes_ask = safe_float(data.get('yes_ask_dollars', 1))
    no_bid = safe_float(data.get('no_bid_dollars', 0))
    no_ask = safe_float(data.get('no_ask_dollars', 1))
    
    print(f"\n✓ Market is accessible")
    print(f"  YES bid/ask: ${yes_bid:.3f} / ${yes_ask:.3f}")
    print(f"  NO bid/ask: ${no_bid:.3f} / ${no_ask:.3f}")
    
    if status != 'open' and status != 'active':
        print(f"\n⚠ WARNING: Market status is '{status}' - may not be tradeable")
except Exception as e:
    print(f"✗ ERROR: Cannot access market: {e}")
    print(f"  The market may be closed, settled, or invalid.")
    print(f"  Please select a different market with status='open'")

✓ Selected Market:
  Ticker: KXNCAAMBGAME-26JAN20UNLVUSU-USU
  Title: UNLV at Utah St. Winner?
  Status: active

✓ Market is accessible
  YES bid/ask: $0.800 / $0.810
  NO bid/ask: $0.190 / $0.200


In [14]:
# Step 2: Create and start the market maker bot
bot = MarketMakerBot(
    client=kalshi_wrapped.GetClient(),
    market=selected_market,       # Pass the market object
    side="yes",                   # "yes" or "no"
    spread_bps=70,               # 2.0% spread (200 basis points)
    size_per_level=1,            # 25 contracts per level
    max_position=6,               # max ±5 contracts inventory
    stop_loss_pct=-1.5,           # stop if return <= -2%
    take_profit_pct=2,          # stop if return >= +3%
    skew_on_inventory=True,       # adjust quotes based on inventory
    skew_bps_per_contract=100,     # 0.1% adjustment per contract
    max_order_age_s=20,
    check_interval_s=0.5,         # update quotes every 3 seconds
    maker_fee_pct=0.0,            # Kalshi maker fee (typically 0%)
    taker_fee_pct=7.0,            # Kalshi taker fee (~7% of profit)
).start()

=== MarketMakerBot started: logs/mm_2026-01-20_b7f64c27.log ===
MarketMakerBot initialized:
  Ticker: KXATPMATCH-26JAN20ZHEMOU-ZHE, Side: yes
  Spread: 70 bps | Size: 1 | MaxPos: ±6
  Stop: -1.50% | Take: 2.00%
  Fees: maker=0.0000% taker=7.0000%
  Skew: True (100 bps/contract)
  max_order_age_s=20.0
Market maker bot started


[22:50:41] FILL BUY    1 @ 0.08 | fee 0.0000 | pos +1 | pnl 0.00 (real 0.00, unr 0.00)
[22:50:43] FILL SELL   1 @ 0.08 | fee 0.0000 | pos +0 | pnl 0.00 (real 0.00, unr 0.00)
[22:50:47] FILL SELL   1 @ 0.09 | fee 0.0000 | pos -1 | pnl 0.00 (real 0.00, unr 0.00)
[22:50:50] FILL BUY    1 @ 0.09 | fee 0.0000 | pos +0 | pnl 0.00 (real 0.00, unr 0.00)
Error getting mid price: (429)
Reason: Too Many Requests
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '68', 'Connection': 'keep-alive', 'Date': 'Wed, 21 Jan 2026 06:50:51 GMT', 'x-content-type-options': 'nosniff', 'content-security-policy': "default-src 'none';", 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 8df3a0d9885300dae74a65758d72215c.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'SFO53-P10', 'X-Amz-Cf-Id': 'PvUb-hFYJIpFA8cwiqqndcj64FjHhxq8TnEr5w9e6Bh9poMkdU4yWg=='})
HTTP response body: {"error":{"code":"too


MARKET MAKER SUMMARY
Ticker: KXATPMATCH-26JAN20ZHEMOU-ZHE (yes)
Total Contracts Filled: 5
Final Position: +1
Peak Margin Used (worst-case loss): $0.91
Total Fees Paid: $0.00
Total P&L (after fees): $-0.41 (-45.05%)
Realized P&L: $0.00
Unrealized P&L (est): $-0.41



In [ ]:
bot.stop()

In [41]:
# Step 3: Monitor the bot status
print(f"Position: {bot.position:+d} contracts")
print(f"Realized P&L: ${bot.realized_pnl:.2f}")
print(f"Unrealized P&L: ${bot._calculate_unrealized_pnl():.2f}")
print(f"Total P&L: ${bot.get_total_pnl():.2f}")
print(f"Total Filled: {bot.total_filled} contracts")
print(f"Active Orders: {len(bot.active_orders)}")

Position: +0 contracts
Realized P&L: $0.00
Unrealized P&L: $0.00
Total P&L: $0.00
Total Filled: 0 contracts
Active Orders: 2


In [42]:
# Step 4: View trade history
if bot.trade_history:
    import pandas as pd
    df = pd.DataFrame(bot.trade_history)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    print("\nTrade History:")
    print(df.to_string(index=False))
else:
    print("No trades executed yet")

No trades executed yet


In [43]:
# Step 5: Stop the bot manually (if needed)
bot.stop()
# The bot will automatically stop when stop_loss or take_profit is hit